# Lesson 6: A byte-pair encoding tokenizer

Implement a small byte-level BPE tokenizer with exact text round trips, including characters absent from the training corpus.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Begin with UTF-8 bytes

Character tokenization gives each character one ID. Byte-level BPE begins with all 256 byte values and repeatedly merges common adjacent pairs. A token can therefore represent part of a character, a whole character, or several characters. This educational tokenizer has no pre-tokenizer or special-token system.


In [ ]:
from collections import Counter

corpus = 'low lower lowest low lower newest newer ' * 8
ids = list(corpus.encode('utf-8'))
vocabulary = {i: bytes([i]) for i in range(256)}
merges = []

def merge_pair(ids, pair, new_id):
    result = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result

print('Original byte count:', len(ids))


## Learn merge rules

Each iteration counts pairs in the current sequence, selects the most frequent pair, and assigns it a new token ID. Apply non-overlapping replacements left to right.


In [ ]:
num_merges = 20
for _ in range(num_merges):
    counts = Counter(zip(ids, ids[1:]))
    if not counts:
        break
    pair, frequency = counts.most_common(1)[0]
    if frequency < 2:
        break
    new_id = len(vocabulary)
    vocabulary[new_id] = vocabulary[pair[0]] + vocabulary[pair[1]]
    merges.append((pair, new_id))
    ids = merge_pair(ids, pair, new_id)
    print(new_id, repr(vocabulary[new_id]), 'frequency:', frequency)
print('Final token count:', len(ids))


## Encode new text and decode exactly

New text starts as bytes and receives the learned merges in rank order. Unseen characters remain representable because the base vocabulary covers every byte. Train a language model on these new IDs to use this tokenizer; old character-model weights are incompatible.


In [ ]:
def bpe_encode(s):
    result = list(s.encode('utf-8'))
    for pair, new_id in merges:
        result = merge_pair(result, pair, new_id)
    return result

def bpe_decode(token_ids):
    return b''.join(vocabulary[i] for i in token_ids).decode('utf-8')

for sample in (corpus, 'lowest newer', 'Hello, café 🌍!', ''):
    encoded = bpe_encode(sample)
    assert bpe_decode(encoded) == sample
    print(repr(sample[:40]), '| bytes:', len(sample.encode('utf-8')), '| tokens:', len(encoded))
print('Vocabulary size:', len(vocabulary))


## Compare sequence lengths

Merges trade a larger vocabulary for shorter sequences. Shorter sequences can reduce attention work, but larger embedding and output matrices cost memory. This naive implementation scans the sequence repeatedly; it is intended for learning on small text.


In [ ]:
sample = 'low lower lowest'
print('Characters:', list(sample))
print('BPE pieces:', [vocabulary[i] for i in bpe_encode(sample)])


## Try it yourself

Change the corpus and rerun from the start. Compare 5 and 20 merges on held-out text. Explain why a token containing several bytes may not be a complete Unicode character on its own.
